# Imports

In [2]:
# if False:
#     !sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet jupyterlab-vim)"
#     !jupyter labextension enable

In [3]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [4]:
import helpers.hgoogle_drive_api as hgodrapi

hgodrapi.install_needed_modules()

import helpers.hllm_cli as hllmcli

hllmcli.install_needed_modules()

Module 'google' is already installed.
Module 'googleapiclient' is already installed.
Module 'gspread' is already installed.
Module 'llm' is already installed.
Module 'tokencost' is already installed.


In [5]:
# Make sure to use the central cache.
!cd /app

In [6]:
import logging

import pandas as pd

# /venv/lib/python3.12/site-packages/gspread_pandas/spread.py:401: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)` .replace("", np.nan)
pd.set_option("future.no_silent_downcasting", True)

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hpandas as hpandas
import helpers.hprint as hprint

#
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

#
_LOG.info("%s", henv.get_system_signature()[0])
hnotebook.config_notebook()

INFO  > cmd='/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-f87133b6-ad57-44ea-b8f2-0446d7c4968b.json'

Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (arm64v8)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/

INFO  # System signature
  # Cont

In [7]:
import gspread
import gspread_pandas

# gspread_pandas.conf.get_config()
print(gspread_pandas.conf.get_config()["project_id"])
print(gspread.__version__)
print(gspread_pandas.__version__)

import helpers.hgoogle_drive_api as hgodrapi

# Get credentials first.
credentials = hgodrapi.get_credentials(
    service_key_path="/home/.config/gspread_pandas/google_secret.json"
)

import ck_marketing.workflows as ckmktwf
import ck_marketing.plugins as ckmktpi

gspread-gp
5.12.4
3.3.0


/app/ck_marketing/workflows/data_loaders_specific.py:14: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [8]:
import importlib

importlib.reload(ckmktwf)
importlib.reload(ckmktpi)
importlib.reload(hpandas)

<module 'helpers.hpandas' from '/app/helpers_root/helpers/hpandas.py'>

# Load Contact data

In [9]:
contact_dfs = []

## GP_LIn_Connections (2024-12-31)

In [10]:
normalize = True
# normalize = False
df_tmp = ckmktwf.get_data_from_GP_LIn_connections(normalize)
print(hpandas.head(df_tmp))
contact_dfs.append(df_tmp)

INFO  Reading data:
  url='https://docs.google.com/spreadsheets/d/19ziUmqbPaUO73cqlJB1F9y-j1Oq98nMzo6wTmzyVnwg'
  file_name='GP_LinkedIn_Connections_2024_12_31'
  tab_name='Sheet1'
shape=(2205, 20)

  hash                                                    origin          origin_timestamp first_name last_name email email_timestamp email_verification email_verification_timestamp  \
0       PB.LIN_Connections_Exports.GP_Lin_Connections_2024_12_31  2024-12-31T02:08:03.137Z  Graham C.      Peck                                                                         
1       PB.LIN_Connections_Exports.GP_Lin_Connections_2024_12_31  2024-12-31T02:08:03.997Z     Justin   Fortier                                                                         

                             linkedin_url                                          job_title linked_timestamp company_name company_domain industry                               city country  \
0     https://linkedin.com/in/grahamcpeck           

In [11]:
ckmktwf.print_contact_df_stats(df_tmp)

,hash,origin,origin_timestamp,first_name,last_name,email,email_timestamp,email_verification,email_verification_timestamp,linkedin_url,job_title,linked_timestamp,company_name,company_domain,industry,city,country,enrichment_timestamp,type,biography
0,,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2024-12-31T02:08:03.137Z,Graham C.,Peck,,,,,https://linkedin.com/in/grahamcpeck,CFO & Partner,,FYC Labs,,,"Chicago, Illinois, United States",,,,Co-Founder @ DealSend & Attaq Vector | Partner...


,0
num_rows,2205
count_no_dups,2200 / 2205 = 99.77%
count_no_ascii,2194 / 2200 = 99.73%
count_email,0 / 2194 = 0.00%
count_email_verification,0 / 2194 = 0.00%
count_name_dups,0 / 2194 = 0.00%
count_origin,2194 / 2194 = 100.00%


## GP_Lin_Connections (2025_12_05)

In [12]:
# The problem is that:
# 1) It's not easy to export all the LinkedIn connections at once
# 2) We have exported in chunks
# 3) Some of the connections were enriched

# The solution is to find the connections that are not enriched.

# GP_LinkedIn_Connections_2024_12_31
# url1 = "https://docs.google.com/spreadsheets/d/19ziUmqbPaUO73cqlJB1F9y-j1Oq98nMzo6wTmzyVnwg/edit?gid=568530034#gid=568530034"
# tab_name1 = "Sheet1"
# df1 = hgodrapi.from_gsheet(url1, tab_name=tab_name1, credentials=credentials)
df1 = ckmktwf.get_data_from_GP_LIn_connections(normalize)
print(df1.shape)

# GP_LinkedIn_Connections_2025_12_05
url2 = "https://docs.google.com/spreadsheets/d/1vz4cYvWOjkIkNQghIhj6DBKUr6OZ7bSEUKNlv0ic-Xk/edit?gid=1753306632#gid=1753306632"
tab_name2 = "Sheet3"
tag = "GP_Lin_Connections_after_2025_12_05"
normalize = True
# df2 = hgodrapi.from_gsheet(url2, tab_name=tab_name2, credentials=credentials)
df2 = ckmktwf.get_data_from_LinkedIn_Connections_Exports(
    url2, tab_name2, tag, normalize
)
print(df2.shape)

(2205, 20)
INFO  Reading data:
  url='https://docs.google.com/spreadsheets/d/1vz4cYvWOjkIkNQghIhj6DBKUr6OZ7bSEUKNlv0ic-Xk/edit?gid=1753306632#gid=1753306632'
  file_name='GP_LinkedIn_Connections_2025_12_05'
  tab_name='Sheet3'
(1424, 20)


In [13]:
df1.head(2)

,hash,origin,origin_timestamp,first_name,last_name,email,email_timestamp,email_verification,email_verification_timestamp,linkedin_url,job_title,linked_timestamp,company_name,company_domain,industry,city,country,enrichment_timestamp,type,biography
0,,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2024-12-31T02:08:03.137Z,Graham C.,Peck,,,,,https://linkedin.com/in/grahamcpeck,CFO & Partner,,FYC Labs,,,"Chicago, Illinois, United States",,,,Co-Founder @ DealSend & Attaq Vector | Partner...
1,,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2024-12-31T02:08:03.997Z,Justin,Fortier,,,,,https://linkedin.com/in/justinffortier,Chief Executive Officer / Chief Technical Officer,,FYC Labs,,,"Folsom, California, United States",,,,Founder + CEO/CTO @ FYC Labs; I love creating ...


In [14]:
df2.head(2)

,hash,origin,origin_timestamp,first_name,last_name,email,email_timestamp,email_verification,email_verification_timestamp,linkedin_url,job_title,linked_timestamp,company_name,company_domain,industry,city,country,enrichment_timestamp,type,biography
0,,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2025-12-09T23:39:42.310Z,Sakshi,Jain,,2025-12-09T23:39:42.310Z,,,https://www.linkedin.com/in/sakshi-jain-33058a...,Business Development Representative | Strategi...,,,,,,,,,
1,,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2025-12-09T23:39:42.310Z,Stuti,Patel,,2025-12-09T23:39:42.310Z,,,https://www.linkedin.com/in/stuti-patel-639473...,"Data Science Grad @ University of Maryland, Co...",,,,,,,,,


In [15]:
print(df1.shape)
print(df2.shape)

(2205, 20)
(1424, 20)


In [16]:
cols = ["first_name", "last_name"]

print(df1[cols].dropna())
print(df2[cols].dropna())

len(set(df2[cols]) - set(df1[cols]))

     first_name         last_name
0     Graham C.              Peck
1        Justin           Fortier
2        Eugene     Gavrilov, PhD
3        Merlin            Yamssi
4         Jaime            Murray
...         ...               ...
2200        Bob            Cramer
2201   Benjamin          Orthlieb
2202       Luis  Llorens Gonzalez
2203     Navita             Goyal
2204    Angelos      Mavrogiannis

[2205 rows x 2 columns]
     first_name   last_name
0        Sakshi        Jain
1         Stuti       Patel
2       Stefano        Oppo
3       Colleen         Qiu
4        Ramani  Duraiswami
...         ...         ...
1419      Navya      Bansal
1420       Karl     Leodler
1421    Vinayak       Dhruv
1422      Parth  Maheshwari
1423     Sriram      Prasad

[1424 rows x 2 columns]


0

In [17]:
diff = df1.merge(
    df2,
    on=cols,
    # how="left",
    how="outer",
    indicator=True,
)

print("common=", (diff[diff["_merge"] == "both"]).shape)
print("df1 only=", (diff[diff["_merge"] == "left_only"]).shape)
print("df2 only=", (diff[diff["_merge"] == "right_only"]).shape)

common= (69, 39)
df1 only= (2136, 39)
df2 only= (1355, 39)


In [18]:
# Find the values that are only df2.
df2_only = df2.merge(
    diff.loc[diff["_merge"].eq("right_only"), cols].drop_duplicates(),
    on=cols,
    how="inner",
)

In [19]:
df2_only.head(3)

,hash,origin,origin_timestamp,first_name,last_name,email,email_timestamp,email_verification,email_verification_timestamp,linkedin_url,job_title,linked_timestamp,company_name,company_domain,industry,city,country,enrichment_timestamp,type,biography
0,,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2025-12-09T23:39:42.310Z,Sakshi,Jain,,2025-12-09T23:39:42.310Z,,,https://www.linkedin.com/in/sakshi-jain-33058a...,Business Development Representative | Strategi...,,,,,,,,,
1,,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2025-12-09T23:39:42.310Z,Stuti,Patel,,2025-12-09T23:39:42.310Z,,,https://www.linkedin.com/in/stuti-patel-639473...,"Data Science Grad @ University of Maryland, Co...",,,,,,,,,
2,,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2025-12-09T23:39:42.310Z,Stefano,Oppo,,2025-12-09T23:39:42.310Z,,,https://www.linkedin.com/in/stefanooppo/,Photographer Videographer,,,,,,,,,


In [20]:
contact_dfs.append(df2)

# Concat all

In [21]:
print(len(contact_dfs))
# Contacts from LinkedIn connections might have no emails.
allow_no_emails = True
contact_df = pd.concat(contact_dfs)
debug = ""
# debug = "duplicated_emails"
# debug = "remove_chinese_names"
# debug = "remove_empty_first_name"
# debug = "clean_company_names"
# debug = "clean_linkedin_emails"
# debug = "clean_linkedin_websites"
contact_df = ckmktwf.clean_up_contact_df(
    contact_df, allow_no_emails=allow_no_emails, debug=debug
)
display(contact_df.head(2))

2


,0
duplicated_emails,0 / 3629 = 0.00%
remove_invalid_emails,0 / 3629 = 0.00%
remove_chinese_names,0 / 3629 = 0.00%
remove_empty_first_names,0 / 3629 = 0.00%
clean_company_names,0 / 3629 = 0.00%
clean_linkedin_nans,0 / 3629 = 0.00%
clean_linkedin_emails,0 / 3629 = 0.00%
clean_linkedin_websites,0 / 3629 = 0.00%
clean_emails,0 / 3629 = 0.00%


,hash,origin,origin_timestamp,first_name,last_name,email,email_timestamp,email_verification,email_verification_timestamp,linkedin_url,job_title,linked_timestamp,company_name,company_domain,industry,city,country,enrichment_timestamp,type,biography
0,f7d75293943e6006b08c989ecf4935aa,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2024-12-31T02:08:03.137Z,Graham C.,Peck,,,,,https://linkedin.com/in/grahamcpeck,CFO & Partner,,FYC Labs,,,"Chicago, Illinois, United States",,,,Co-Founder @ DealSend & Attaq Vector | Partner...
1,ae46812c1d03572ae492d1ba88cc8148,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2024-12-31T02:08:03.997Z,Justin,Fortier,,,,,https://linkedin.com/in/justinffortier,Chief Executive Officer / Chief Technical Officer,,FYC Labs,,,"Folsom, California, United States",,,,Founder + CEO/CTO @ FYC Labs; I love creating ...


In [22]:
contact_df.iloc[0]

hash                                             f7d75293943e6006b08c989ecf4935aa
origin                          PB.LIN_Connections_Exports.GP_Lin_Connections_...
origin_timestamp                                         2024-12-31T02:08:03.137Z
first_name                                                              Graham C.
last_name                                                                    Peck
email                                                                            
email_timestamp                                                                  
email_verification                                                               
email_verification_timestamp                                                     
linkedin_url                                  https://linkedin.com/in/grahamcpeck
job_title                                                           CFO & Partner
linked_timestamp                                                                 
company_name    

In [23]:
ckmktwf.print_contact_df_detailed_stats(contact_df)

,valid [pct],unique [pct],invalid [pct],empty [pct],nan [pct]
col_name,,,,,
hash,100.000000,100.000000,0.000000,0.000000,0.000000
origin,100.000000,0.060000,0.000000,0.000000,0.000000
origin_timestamp,100.000000,62.170000,0.000000,0.000000,0.000000
first_name,100.000000,59.850000,0.000000,0.000000,0.000000
last_name,100.000000,82.090000,0.000000,0.000000,0.000000
email,0.000000,0.030000,100.000000,100.000000,0.000000
email_timestamp,39.240000,1.430000,60.760000,60.760000,0.000000
email_verification,0.000000,0.030000,100.000000,100.000000,0.000000
email_verification_timestamp,0.000000,0.030000,100.000000,100.000000,0.000000


In [24]:
ckmktwf.sanity_check_contact_df(contact_df)

{'nan': 1.0}


In [25]:
ckmktwf.print_contact_df_stats(contact_df)

,hash,origin,origin_timestamp,first_name,last_name,email,email_timestamp,email_verification,email_verification_timestamp,linkedin_url,job_title,linked_timestamp,company_name,company_domain,industry,city,country,enrichment_timestamp,type,biography
0,f7d75293943e6006b08c989ecf4935aa,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2024-12-31T02:08:03.137Z,Graham C.,Peck,,,,,https://linkedin.com/in/grahamcpeck,CFO & Partner,,FYC Labs,,,"Chicago, Illinois, United States",,,,Co-Founder @ DealSend & Attaq Vector | Partner...


,0
num_rows,3629
count_no_dups,3554 / 3629 = 97.93%
count_no_ascii,3539 / 3554 = 99.58%
count_email,0 / 3539 = 0.00%
count_email_verification,0 / 3539 = 0.00%
count_name_dups,0 / 3539 = 0.00%
count_origin,3539 / 3539 = 100.00%


In [26]:
contact_df["origin"].value_counts()

origin
PB.LIN_Connections_Exports.GP_Lin_Connections_2024_12_31          2205
PB.LIN_Connections_Exports.GP_Lin_Connections_after_2025_12_05    1424
Name: count, dtype: int64

In [27]:
# ckmktwf.save_to_gsheet(contact_df)

## Clean up names

In [28]:
contact_df = ckmktpi.clean_and_track_name_changes(contact_df)

debug_df = ckmktpi.get_debug_clean_name_df(contact_df)
ckmktpi.get_clean_name_stats(contact_df)

,first_name,last_name,cleaned_first_name,first_alias,cleaned_last_name,second_alias,is_modified
0,Graham C.,Peck,Graham,,Peck,,True
1,Justin,Fortier,Justin,,Fortier,,False
2,Eugene,"Gavrilov, PhD",Eugene,,Gavrilov,,True


,0
is_modified,426 / 3629 = 11.74%
empty_first_name,1 / 3629 = 0.03%
empty_last_name,7 / 3629 = 0.19%
has_alias,39 / 3629 = 1.07%


In [29]:
hpandas.filter_df(debug_df, "is_modified", True).head(10)

INFO  selected=426 / 3629 = 11.74%


,first_name,last_name,cleaned_first_name,first_alias,cleaned_last_name,second_alias,is_modified
0,Graham C.,Peck,Graham,,Peck,,True
2,Eugene,"Gavrilov, PhD",Eugene,,Gavrilov,,True
17,Bilal,"Ayyub, Ph.D., P.E., Dist.M.ASCE, Hon.M.ASME",Bilal,,Ayyub,,True
18,IRINA,MURESANU,Irina,,Muresanu,,True
43,Marcin,Sołtysiak,Marcin,,Sotysiak,,True
45,Hitesh,kyatham,Hitesh,,Kyatham,,True
59,Hao-Lin (Alex),Chiang,Hao-Lin,Alex,Chiang,,True
78,Saurabh,"Sachdeva, Ph.D.",Saurabh,,Sachdeva,,True
93,Carlos,"Jaureguizar, PhD",Carlos,,Jaureguizar,,True
135,Maya,Bakhai 🌶,Maya,,Bakhai,,True


In [30]:
contact_df = ckmktpi.merge_clean_names_df(contact_df)

contact_df.head(1)

,hash,origin,origin_timestamp,first_name,last_name,email,email_timestamp,email_verification,email_verification_timestamp,linkedin_url,job_title,linked_timestamp,company_name,company_domain,industry,city,country,enrichment_timestamp,type,biography
0,f7d75293943e6006b08c989ecf4935aa,PB.LIN_Connections_Exports.GP_Lin_Connections_...,2024-12-31T02:08:03.137Z,Graham,Peck,,,,,https://linkedin.com/in/grahamcpeck,CFO & Partner,,FYC Labs,,,"Chicago, Illinois, United States",,,,Co-Founder @ DealSend & Attaq Vector | Partner...


In [31]:
contact_df.shape

(3629, 20)

In [33]:
# hgodrapi.save_df_to_tmp_gsheet(contact_df)

# Classify

In [34]:
df = contact_df.head(500)

In [35]:
# hgodrapi.save_df_to_tmp_gsheet(df, remove_empty_columns=True)
url = "https://docs.google.com/spreadsheets/d/1y7A__hpV8n9nyQpNLDso6fGVsjBu5LDfgx1xYU6cwOI/edit?gid=0#gid=0"
hgodrapi.save_df_to_tmp_gsheet(df, url=url, tab_name="before")

INFO  Data written to:
tab 'before'
Google Sheet 'https://docs.google.com/spreadsheets/d/1y7A__hpV8n9nyQpNLDso6fGVsjBu5LDfgx1xYU6cwOI/edit?gid=0#gid=0'
INFO  url=https://docs.google.com/spreadsheets/d/1y7A__hpV8n9nyQpNLDso6fGVsjBu5LDfgx1xYU6cwOI/edit?gid=1094411348#gid=1094411348


In [ ]:
hllmcli.shutup_llm_logging()

In [37]:
model = "gpt-4o-mini"
df = ckmktwf.classify_industry_type_executive(df, model=model)

INFO  Classifying industry, department, and executive type of 500 items
INFO  batch_size=50, model='gpt-4o-mini'
INFO  Processing 500 items in 10 batches of 50 items each
INFO  model='gpt-4o-mini', batch_mode='combined'


Classifying industry:   0%|                                                                                   …

INFO  Processing completed:
{'elapsed_time_in_seconds': 56.01050686836243,
 'num_batches': 10,
 'num_items': 500,
 'num_skipped': 5,
 'total_cost_in_dollars': 0.0035953499999999998}
INFO  stats={'num_items': 500, 'num_skipped': 5, 'num_batches': 10, 'total_cost_in_dollars': 0.0035953499999999998, 'elapsed_time_in_seconds': 56.01050686836243}
INFO  Processing 500 items in 10 batches of 50 items each
INFO  model='gpt-4o-mini', batch_mode='combined'


Classifying department:   0%|                                                                                 …

INFO  Processing completed:
{'elapsed_time_in_seconds': 74.00314450263977,
 'num_batches': 10,
 'num_items': 500,
 'num_skipped': 0,
 'total_cost_in_dollars': 0.0168402}
INFO  stats={'num_items': 500, 'num_skipped': 0, 'num_batches': 10, 'total_cost_in_dollars': 0.0168402, 'elapsed_time_in_seconds': 74.00314450263977}
INFO  Processing 500 items in 10 batches of 50 items each
INFO  model='gpt-4o-mini', batch_mode='combined'


...

,count,pct [%]
type,,
VC,146,29.2
unknown,108,21.6
Student,64,12.8
Technology,42,8.4
Professor,35,7.0
"Finance, Legal & Risk",22,4.4
Corporate Development,20,4.0
Angel Investor,19,3.8
Product,10,2.0


INFO  # industry


,count,pct [%]
industry,,
unknown,165,33.0
Education,109,21.8
Financial Services,105,21.0
IT - Software,34,6.8
IT - Consulting & Integration,11,2.2
Healthcare,9,1.8
IT - Data & Analytics,9,1.8
Government & Nonprofits,8,1.6
IT - Cybersecurity,5,1.0


INFO  # executive


,count,pct [%]
executive,,
unknown,314,62.8
Partner,72,14.4
CEO,36,7.2
Founder,14,2.8
Principal,9,1.8
Managing Partner,8,1.6
Co-Founder,7,1.4
,5,1.0
CFO,4,0.8


In [38]:
hpandas.display_value_counts_stats_df(
    df,
    col_names=["type", "industry", "executive"],
    num_rows=10,  # Number of top rows to return
)

INFO  # type


,count,pct [%]
type,,
VC,146,29.2
unknown,108,21.6
Student,64,12.8
Technology,42,8.4
Professor,35,7.0
"Finance, Legal & Risk",22,4.4
Corporate Development,20,4.0
Angel Investor,19,3.8
Product,10,2.0


INFO  # industry


,count,pct [%]
industry,,
unknown,165,33.0
Education,109,21.8
Financial Services,105,21.0
IT - Software,34,6.8
IT - Consulting & Integration,11,2.2
Healthcare,9,1.8
IT - Data & Analytics,9,1.8
Government & Nonprofits,8,1.6
IT - Cybersecurity,5,1.0


INFO  # executive


,count,pct [%]
executive,,
unknown,314,62.8
Partner,72,14.4
CEO,36,7.2
Founder,14,2.8
Principal,9,1.8
Managing Partner,8,1.6
Co-Founder,7,1.4
,5,1.0
CFO,4,0.8


In [40]:
# hpandas.remove_empty_columns(hpandas.filter_df(df, "type", "unknown"))

In [ ]:
df.head(2)

In [41]:
hgodrapi.save_df_to_tmp_gsheet(df, remove_empty_columns=True)

INFO  kept 13 / 21 = 61.90% columns: : (13) hash origin origin_timestamp first_name last_name linkedin_url job_title company_name industry city type biography executive

INFO  removed 8 / 21 = 38.10% columns: : (8) email email_timestamp email_verification email_verification_timestamp linked_timestamp company_domain country enrichment_timestamp

INFO  Data written to:
tab 'Sheet13'
Google Sheet 'https://docs.google.com/spreadsheets/d/1NLY7dTmkXmllYfewDH53z-uSRpC9-zBTTmAOB_O30DI/edit?gid=0#gid=0'
INFO  url=https://docs.google.com/spreadsheets/d/1NLY7dTmkXmllYfewDH53z-uSRpC9-zBTTmAOB_O30DI/edit?gid=1365249664#gid=1365249664


# Load into DB

In [ ]:
assert 0

In [ ]:
db_path = "test.sql"

In [ ]:
ckmktwf.print_table_schema(db_path, tables=["Contact"])

In [ ]:
ckmktwf.get_table_count(db_path, "Contact")

In [ ]:
crm_df = ckmktwf.get_as_df(db_path, "Contact")

hpandas.head(crm_df)

In [ ]:
crm_df.origin.unique()

In [ ]:
# mode = "assume_no_overlap"
mode = "assume_idempotent"
ckmktwf.insert_contact_df(db_path, contact_df, mode)

In [ ]:
crm_df = ckmktwf.get_as_df(db_path, "Contact")

print(hpandas.head(crm_df))
print(crm_df.origin.unique())

ckmktwf.get_table_count(db_path, "Contact")